# A.6. Nearby time delays

Fix $\tau_1=300$ and set $\tau_2=300+\Delta\tau$, with $\Delta\tau=100,20,10,5$. Compare dual-delay systems with matched single-delay controls using one trajectory and 20 ESNs per condition.


In [ ]:
import gc
import hashlib
import json
import os
import pickle
import tempfile
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
%run ./ESN.ipynb
%run ./Reconstruct.ipynb
N_x = 4
N_dim = 3 * N_x
N_candidates = 6
N_dim_segments = N_candidates * N_dim
N_train = 10000
N_washout = 100
N_test = 2000
N_units = 7200
N_trajectories = 1
N_reservoir = 20
N_rep = 30
N_evo = 5
TAU_1 = 300
DELTA_VALUES = np.array([100, 20, 10, 5], dtype=int)
MODES = ('dual', 'single')
TRUE_EFFECT = 0
TRUE_CAUSE = 1
SIGMA_IN = 0.5
SPECTRAL_RADIUS = 0.9
TIKHONOV = 0.0001
J_THRESHOLD = 0.01
COUPLING = 0.3
SIGMA_DYN = 1e-06
SIGMA_OBS = 1e-06
DT = 0.01
BURN_IN = 2000
PARAMETER_SEED = 99
INITIAL_SEED = 199
DYNAMICAL_NOISE_SEED = 299
OBSERVATION_NOISE_SEED = 399
OFFSET_SEED = 499
RESERVOIR_SEED = 599
candidate_lags = {int(delta): np.array([0, TAU_1 - delta, TAU_1, TAU_1 + delta // 2, TAU_1 + delta, TAU_1 + 2 * delta], dtype=int) for delta in DELTA_VALUES}
MAX_LAG = max((int(lags.max()) for lags in candidate_lags.values()))
HISTORY_LENGTH = MAX_LAG + 1
RAW_ORIGIN = HISTORY_LENGTH + BURN_IN + MAX_LAG
TRAIN_STOP = N_washout + N_train
TEST_START = TRAIN_STOP + 1 + MAX_LAG
EMBEDDED_LENGTH = TEST_START + N_test

experiment_signature = {'dependencies': {name: hashlib.sha256(Path(name).read_bytes()).hexdigest() for name in ('ESN.ipynb', 'Reconstruct.ipynb')}, 'numpy_version': np.__version__, 'N_x': N_x, 'N_train': N_train, 'N_washout': N_washout, 'N_test': N_test, 'N_units': N_units, 'N_trajectories': N_trajectories, 'N_reservoir': N_reservoir, 'N_rep': N_rep, 'N_evo': N_evo, 'tau_1': TAU_1, 'deltas': DELTA_VALUES.tolist(), 'modes': MODES, 'candidate_lags': {str(delta): lags.tolist() for (delta, lags) in candidate_lags.items()}, 'true_effect': TRUE_EFFECT, 'true_cause': TRUE_CAUSE, 'sigma_in': SIGMA_IN, 'spectral_radius': SPECTRAL_RADIUS, 'tikhonov': TIKHONOV, 'j_threshold': J_THRESHOLD, 'coupling': COUPLING, 'sigma_dyn': SIGMA_DYN, 'sigma_obs': SIGMA_OBS, 'dt': DT, 'burn_in': BURN_IN, 'seeds': [PARAMETER_SEED, INITIAL_SEED, DYNAMICAL_NOISE_SEED, OBSERVATION_NOISE_SEED, OFFSET_SEED, RESERVOIR_SEED]}
signature_token = hashlib.sha256(json.dumps(experiment_signature, sort_keys=True).encode('utf-8')).hexdigest()
RESULT_DIR = Path('result') / 'A.6'
CACHE_DIR = RESULT_DIR / 'runs' / signature_token[:20]
DATA_DIR = CACHE_DIR / 'data'
RUN_DIR = CACHE_DIR / 'runs'
RESULT_PATH = RESULT_DIR / 'A.6.pkl'
FIGURE_DIR = Path('result') / 'figure'
for directory in (DATA_DIR, RUN_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def save_pickle_atomic(value, path):
    path = Path(path)
    (descriptor, temporary_path) = tempfile.mkstemp(dir=path.parent, suffix='.tmp')
    with os.fdopen(descriptor, 'wb') as stream:
        pickle.dump(value, stream, protocol=pickle.HIGHEST_PROTOCOL)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary_path, path)
offset_rng = np.random.RandomState(OFFSET_SEED)
offsets = offset_rng.randint(N_washout, N_test - N_evo, size=(N_trajectories, N_reservoir, N_rep))
parameter_rng = np.random.RandomState(PARAMETER_SEED)
heterogeneity = 2.0 * (parameter_rng.rand(N_x) - 0.5) * 0.06


## Part 1. Observations

Use six candidate delays per condition. Load cached observations or generate missing data. Normalize using the training-set median and interquartile range.


In [ ]:
def adjacency_for_condition(delta, mode):
    lags = candidate_lags[int(delta)]
    adjacency = np.zeros((N_candidates, N_x, N_x), dtype=float)
    first_index = int(np.flatnonzero(lags == TAU_1)[0])
    if mode == 'dual':
        second_index = int(np.flatnonzero(lags == TAU_1 + delta)[0])
        adjacency[first_index, TRUE_EFFECT, TRUE_CAUSE] = 0.5
        adjacency[second_index, TRUE_EFFECT, TRUE_CAUSE] = 0.5
    elif mode == 'single':
        adjacency[first_index, TRUE_EFFECT, TRUE_CAUSE] = 1.0
    return adjacency

def lorenz_rhs(states, adjacency, noise):
    current = states[0].reshape(N_x, 3)
    coupling = np.einsum('lec,lc->e', adjacency, states[:, 1::3]) - current[:, 1] * np.sum(adjacency, axis=(0, 2))
    derivative = np.empty_like(current)
    derivative[:, 0] = -10.0 * (current[:, 0] - current[:, 1] + COUPLING * coupling)
    derivative[:, 1] = 28.0 * (1.0 + heterogeneity) * current[:, 0] - current[:, 1] - current[:, 0] * current[:, 2]
    derivative[:, 2] = -8.0 / 3.0 * current[:, 2] + current[:, 0] * current[:, 1]
    return derivative.reshape(-1) + noise

def generate_raw_lorenz(lags, adjacency, trajectory):
    initial_rng = np.random.RandomState(INITIAL_SEED + trajectory)
    noise_rng = np.random.RandomState(DYNAMICAL_NOISE_SEED + trajectory)
    raw = np.zeros((RAW_ORIGIN + EMBEDDED_LENGTH, N_dim), dtype=float)
    raw[:HISTORY_LENGTH] = initial_rng.rand(N_dim)
    historical_lags = lags[1:]
    for index in range(HISTORY_LENGTH - 1, len(raw) - 1):
        historical_start = raw[index - historical_lags]
        historical_end = raw[index - historical_lags + 1]
        historical_midpoint = (historical_start + historical_end) / 2.0
        noise = noise_rng.normal(0.0, SIGMA_DYN, size=(4, N_dim))
        states = np.vstack((raw[index], historical_start))
        first = lorenz_rhs(states, adjacency, noise[0])
        states = np.vstack((raw[index] + DT * first / 2.0, historical_midpoint))
        second = lorenz_rhs(states, adjacency, noise[1])
        states = np.vstack((raw[index] + DT * second / 2.0, historical_midpoint))
        third = lorenz_rhs(states, adjacency, noise[2])
        states = np.vstack((raw[index] + DT * third, historical_end))
        fourth = lorenz_rhs(states, adjacency, noise[3])
        raw[index + 1] = raw[index] + DT * (first + 2.0 * second + 2.0 * third + fourth) / 6.0
    return raw

def dataset_path(delta, mode, trajectory):
    return DATA_DIR / f'gap_{delta}_{mode}_trajectory_{trajectory:02d}.pkl'

for delta in DELTA_VALUES:
    delta = int(delta)
    lags = candidate_lags[delta]
    for mode in MODES:
        for trajectory in range(N_trajectories):
            path = dataset_path(delta, mode, trajectory)
            if path.exists():
                continue
            adjacency = adjacency_for_condition(delta, mode)
            raw = generate_raw_lorenz(lags, adjacency, trajectory)
            calibration_start = RAW_ORIGIN + N_washout
            calibration_stop = RAW_ORIGIN + TRAIN_STOP
            quartiles = np.percentile(raw[calibration_start:calibration_stop], [25, 50, 75], axis=0, method='midpoint')
            median = quartiles[1]
            iqr = quartiles[2] - quartiles[0]
            observed = (raw - median) / iqr
            observation_rng = np.random.RandomState(OBSERVATION_NOISE_SEED + trajectory)
            if SIGMA_OBS != 0:
                observed += observation_rng.normal(0.0, SIGMA_OBS, size=observed.shape)
            embedded = np.concatenate([observed[RAW_ORIGIN - delay:RAW_ORIGIN - delay + EMBEDDED_LENGTH] for delay in lags], axis=1)
            bundle = {'signature_token': signature_token, 'delta': delta, 'mode': mode, 'trajectory': trajectory, 'candidate_lags': lags.copy(), 'adjacency': adjacency, 'true_delay_mask': adjacency[:, TRUE_EFFECT, TRUE_CAUSE] > 0, 'X_raw': raw, 'X_segments': embedded, 'median': median, 'iqr': iqr, 'calibration_raw_bounds': (calibration_start, calibration_stop), 'raw_origin': RAW_ORIGIN, 'train_stop': TRAIN_STOP, 'test_start': TEST_START}
            save_pickle_atomic(bundle, path)
            del bundle, raw, observed, embedded


## Part 2. DDI

Evaluate 30 histories per ESN over five evolution steps. Reuse completed model results and average DDI across the 20 ESNs.


In [ ]:
def calculate_delay_ddi(esn, test_data, paired_offsets):
    score_sums = np.zeros(N_candidates, dtype=float)
    positive_counts = np.zeros(N_candidates, dtype=int)
    for current_offset in paired_offsets:
        history = test_data[current_offset - N_washout:current_offset]
        state = esn.open_loop(history, np.zeros(esn.n_units))[-1]
        control = esn.evolve(state, N_evo)
        for candidate_index in range(N_candidates):
            target_effect = 3 * TRUE_EFFECT
            target_cause = 3 * TRUE_CAUSE + N_dim * candidate_index + 1
            source_mean = np.mean(control[:, target_cause])
            if abs(source_mean) < J_THRESHOLD:
                continue
            intervention = esn.evolve_edge_removal(state, target_cause, target_effect, N_evo)
            difference = control[:, target_effect] - intervention[:, target_effect]
            score = np.sum(difference ** 2) / N_evo / source_mean
            if score > 0:
                score_sums[candidate_index] += score
                positive_counts[candidate_index] += 1
    scores = np.divide(score_sums, positive_counts, out=np.zeros_like(score_sums), where=positive_counts > 0)
    return (scores, positive_counts)
run_shape = (len(DELTA_VALUES), len(MODES), N_trajectories, N_reservoir)
DDI_scores = np.full(run_shape + (N_candidates,), np.nan)
positive_counts_all = np.zeros(DDI_scores.shape, dtype=int)
elapsed_all = np.full(run_shape, np.nan)
completed = np.zeros(run_shape, dtype=bool)

def run_path(gap_index, mode_index, trajectory, reservoir):
    return RUN_DIR / f'gap_{DELTA_VALUES[gap_index]}_{MODES[mode_index]}_trajectory_{trajectory:02d}_reservoir_{reservoir:02d}.pkl'

def apply_run(payload, indices):
    DDI_scores[indices] = payload['DDI']
    positive_counts_all[indices] = payload['positive_counts']
    elapsed_all[indices] = payload['elapsed_seconds']
    completed[indices] = True

for indices in np.ndindex(run_shape):
    path = run_path(*indices)
    if path.exists():
        with path.open('rb') as input_file:
            payload = pickle.load(input_file)
        if payload['status'] == 'ok':
            apply_run(payload, indices)

for reservoir in range(N_reservoir):
    pending = [indices for indices in np.ndindex(run_shape) if indices[3] == reservoir and not completed[indices]]
    if not pending:
        continue
    np.random.seed(RESERVOIR_SEED + reservoir)
    esn = ESN(N_units, N_dim_segments, N_dim_segments, sigma_in=SIGMA_IN)
    esn.rho = SPECTRAL_RADIUS
    esn.tikh = TIKHONOV
    for indices in pending:
        gap_index, mode_index, trajectory, reservoir_index = indices
        delta = int(DELTA_VALUES[gap_index])
        mode = MODES[mode_index]
        started = time.perf_counter()
        with dataset_path(delta, mode, trajectory).open('rb') as input_file:
            bundle = pickle.load(input_file)
        embedded = np.asarray(bundle['X_segments'])
        esn.train(embedded[:N_washout], embedded[N_washout:TRAIN_STOP - 1], embedded[N_washout + 1:TRAIN_STOP])
        scores, counts = calculate_delay_ddi(esn, embedded[TEST_START:TEST_START + N_test], offsets[trajectory, reservoir_index])
        payload = {'signature_token': signature_token, 'indices': indices, 'status': 'ok', 'DDI': scores, 'positive_counts': counts, 'elapsed_seconds': time.perf_counter() - started}
        save_pickle_atomic(payload, run_path(*indices))
        apply_run(payload, indices)
        del bundle, embedded
    del esn
    gc.collect()

saved_results = {'signature_token': signature_token, 'experiment_signature': experiment_signature, 'delta_values': DELTA_VALUES.copy(), 'modes': MODES, 'candidate_lags': candidate_lags, 'DDI_scores': DDI_scores, 'positive_counts': positive_counts_all, 'elapsed_seconds': elapsed_all, 'offsets': offsets, 'heterogeneity': heterogeneity, 'data_directory': str(DATA_DIR)}
save_pickle_atomic(saved_results, RESULT_PATH)


## Part 3. DDI curves

Save a 2-by-2 PNG figure. Lines show the mean across 20 ESNs; shaded bands show one standard deviation.


In [ ]:
with RESULT_PATH.open('rb') as input_file:
    saved_results = pickle.load(input_file)
if saved_results['signature_token'] != signature_token:
    raise ValueError('Results do not match the current parameters.')
scores = np.asarray(saved_results['DDI_scores'], dtype=float)
ddi_mean = np.mean(scores[:, :, 0], axis=2)
ddi_std = np.std(scores[:, :, 0], axis=2, ddof=1)
dual_index = MODES.index('dual')
single_index = MODES.index('single')
styles = ((dual_index, '#0072B2', 'o', '-', 'Two true delays'), (single_index, '#D55E00', 's', '--', 'Single-delay control'))

def save_figure(figure, stem):
    png_path = FIGURE_DIR / f'{stem}.png'
    figure.savefig(png_path, dpi=600, facecolor='white')
with plt.rc_context({'font.family': 'serif', 'font.serif': ['Times New Roman'], 'font.size': 13, 'mathtext.fontset': 'stix'}):
    ddi_figure = plt.figure(figsize=(13.2, 9.6), facecolor='white')
    layout = ddi_figure.add_gridspec(2, 2)
    ddi_figure.subplots_adjust(left=0.085, right=0.98, bottom=0.085, top=0.94, wspace=0.24, hspace=0.42)
    ddi_axes = np.array([[ddi_figure.add_subplot(layout[row, column]) for column in range(2)] for row in range(2)])
    candidate_positions = np.arange(N_candidates)
    for (gap_index, (axes, delta)) in enumerate(zip(ddi_axes.flat, DELTA_VALUES)):
        delta = int(delta)
        for (mode_index, color, marker, linestyle, label) in styles:
            mean = ddi_mean[gap_index, mode_index]
            spread = ddi_std[gap_index, mode_index]
            axes.plot(candidate_positions, mean, color=color, marker=marker, markersize=6, linewidth=2.0, linestyle=linestyle, label=label)
            axes.fill_between(candidate_positions, np.maximum(mean - spread, 0), mean + spread, color=color, alpha=0.15, linewidth=0)
        axes.set_xticks(candidate_positions, [str(int(delay)) for delay in candidate_lags[delta]])
        axes.set_xlabel('Time delay', fontsize=12)
        axes.set_ylabel('DDI', fontsize=15)
        axes.set_title(f'$\\Delta\\tau={delta},\\quad\\tau_1={TAU_1},\\quad\\tau_2={TAU_1 + delta}$', fontsize=15, pad=13)
        axes.set_xlim(-0.2, N_candidates - 0.8)
        data_top = max(float(np.max(ddi_mean[gap_index] + ddi_std[gap_index])), 1e-12)
        axes.set_ylim(0, 1.55 * data_top)
        axes.yaxis.set_major_locator(MaxNLocator(nbins=4))
        axes.ticklabel_format(axis='y', style='sci', scilimits=(0, 0), useMathText=True)
        axes.yaxis.get_offset_text().set_fontsize(10)
        axes.tick_params(axis='both', labelsize=11, width=1.0, direction='out')
        axes.grid(False)
        for spine in axes.spines.values():
            spine.set_linewidth(1.0)
        axes.legend(loc='upper left', bbox_to_anchor=(0.015, 0.985), frameon=False, fontsize=10, handlelength=2.4)
    save_figure(ddi_figure, 'A.6_DDI_2x2')
    plt.show()
import csv
summary_path = RESULT_DIR / 'A.6_summary.csv'
with summary_path.open('w', newline='', encoding='utf-8-sig') as output_file:
    writer = csv.writer(output_file)
    writer.writerow(['separation_samples', 'mode', 'candidate_delay_samples', 'is_true_delay', 'DDI_mean', 'DDI_SD_across_reservoirs'])
    for (gap_index, delta) in enumerate(DELTA_VALUES):
        for (mode_index, mode) in enumerate(MODES):
            for (candidate_index, delay) in enumerate(candidate_lags[int(delta)]):
                is_true_delay = delay == TAU_1 or (mode == 'dual' and delay == TAU_1 + int(delta))
                writer.writerow([int(delta), mode, int(delay), is_true_delay, ddi_mean[gap_index, mode_index, candidate_index], ddi_std[gap_index, mode_index, candidate_index]])
print(FIGURE_DIR)
